## Conditional MusicVAE — Kaggle-ready training

This notebook trains the **Conditional VAE** (bidirectional Transformer encoder → global latent $z_p$ → GRU conductor → bar latents $z_k$ → FiLM causal Transformer decoder).

### What you need to do on Kaggle
- Attach your memmap datasets as Kaggle Datasets.
- Your layout is expected to be:
  - `/kaggle/input/datasets/<user>/<preprocessed-memmaptrain>/train/...`
  - `/kaggle/input/datasets/<user>/<preprocessed-memmapval>/val/...`
- Set `KAGGLE_USER`, `TRAIN_DATASET`, and `VAL_DATASET` below.
- Click **Run All**.

### Outputs
- Checkpoints and logs are written to `/kaggle/working/`.
- Logs include posterior-collapse diagnostics (KL bits/dim, active units, recon delta during $\beta=0$ portion of cycle).

In [ ]:
from __future__ import annotations

import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import torch

print('cwd:', os.getcwd())
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('torch cuda:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
print('input exists:', INPUT_ROOT.exists())
print('working exists:', WORKING_ROOT.exists())

# List datasets attached to the notebook
if INPUT_ROOT.exists():
    print('--- /kaggle/input ---')
    for p in sorted(INPUT_ROOT.iterdir()):
        if p.is_dir():
            print(' -', p.name)


In [ ]:
# -------------------------
# Config (edit these on Kaggle)
# -------------------------

# Your Kaggle layout is: /kaggle/input/datasets/<user>/<dataset-slug>/...
KAGGLE_USER = "karandeepshoker"
TRAIN_DATASET = "preprocessed-memmaptrain"
VAL_DATASET = "preprocessed-memmapval"
# TEST_DATASET = "preprocessed-memmaptest"  # optional

# Inside each dataset, where do the split folders live?
# Set to Path('.') if the dataset contains train/ or val/ at its root.
# Set to Path('preprocessed_memmap') if it contains preprocessed_memmap/train, etc.
SPLIT_ROOT_REL = Path('.')

MAX_STEPS = 10_000  # optimizer updates
BATCH_SIZE = 8
GRAD_ACCUM = 2  # accumulate this many micro-batches per optimizer update
LR = 3e-4
WEIGHT_DECAY = 0.01
ETA_MIN = 1e-5

EVAL_EVERY = 500
EVAL_BATCHES = 50
LOG_EVERY = 10  # log train loss every N steps
SAVE_EVERY = 500

NUM_WORKERS = 2
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4

# Best-effort de-correlation: i.i.d-ish sampling with replacement
USE_REPLACEMENT_SAMPLER = True

# Spec β schedule (cyclical)
BETA_MAX = 0.1
BETA_CYCLE_LEN = 10_000
BETA_RAMP_LEN = 5_000

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = True and (DEVICE == 'cuda')

OUT_DIR = Path('/kaggle/working/vae_run')
CKPT_PATH = OUT_DIR / 'ckpt.pt'
RESUME_PATH = None  # set to CKPT_PATH to resume, or leave None to start fresh

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('DEVICE:', DEVICE, 'AMP:', USE_AMP)
print('OUT_DIR:', OUT_DIR)


In [ ]:
# -------------------------
# Self-contained code (write a tiny module file)
# -------------------------

MODULE_PATH = Path('musicgen_kaggle_vae_min.py')

# IMPORTANT: The VAE code block below is copied verbatim from ml/src/musicgen/models/vae.py.
VAE_CODE = r'''
from __future__ import annotations

from dataclasses import dataclass
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class VAEConfig:
    vocab_size: int = 195
    block_size: int = 1024

    # Encoder (bidirectional)
    enc_n_layers: int = 6
    enc_n_embed: int = 512
    enc_n_head: int = 8
    enc_d_ff: int = 2048
    z_dim: int = 128
    enc_dropout: float = 0.1

    # Attributes
    bars_per_sample: int = 8
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32  # 4*32 = 128

    # Conductor
    cond_hidden: int = 384
    cond_layers: int = 2

    # Decoder (causal)
    dec_n_layers: int = 6
    dec_n_embed: int = 512
    dec_n_head: int = 8
    dec_d_ff: int = 2048
    dec_dropout: float = 0.1


class AttributeEmbedder(nn.Module):
    """
    attributes: LongTensor [B, bars, 4] with values 0..7
    returns:    FloatTensor [B, bars, 128]
    """

    def __init__(self, cfg: VAEConfig):
        super().__init__()
        assert cfg.num_attributes == 4
        self.embeds = nn.ModuleList(
            [nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)]
        )

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        return torch.cat([a0, a1, a2, a3], dim=-1)  # [B, bars, 128]


def broadcast_by_bar_indices(bars_x_c: torch.Tensor, bar_indices: torch.Tensor) -> torch.Tensor:
    """
    bars_x_c:  [B, bars, C]
    bar_indices: [B, T] values in 0..bars-1
    returns:   [B, T, C]
    """
    B, bars, C = bars_x_c.shape
    idx = bar_indices.to(dtype=torch.long).clamp_(0, bars - 1)
    return bars_x_c.gather(1, idx.unsqueeze(-1).expand(B, idx.size(1), C))


class BiTransformerEncoder(nn.Module):
    """
    Bidirectional encoder with learned [CLS] token, outputs mu/logvar for z_p.
    """

    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.enc_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size + 1, cfg.enc_n_embed)  # +1 for CLS position
        self.cls = nn.Parameter(torch.zeros(cfg.enc_n_embed))

        layer = nn.TransformerEncoderLayer(
            d_model=cfg.enc_n_embed,
            nhead=cfg.enc_n_head,
            dim_feedforward=cfg.enc_d_ff,
            dropout=float(cfg.enc_dropout),
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.enc_n_layers)
        self.ln_f = nn.LayerNorm(cfg.enc_n_embed)

        self.mu = nn.Linear(cfg.enc_n_embed, cfg.z_dim)
        self.logvar = nn.Linear(cfg.enc_n_embed, cfg.z_dim)

        nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, X: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Encoder expects block_size={self.cfg.block_size}, got T={T}")

        tok = self.tok_emb(X)  # [B,T,C]
        cls = self.cls.view(1, 1, -1).expand(B, 1, -1)  # [B,1,C]
        h = torch.cat([cls, tok], dim=1)  # [B,T+1,C]

        pos = torch.arange(T + 1, device=X.device)
        h = h + self.pos_emb(pos)[None, :, :]

        h = self.encoder(h)
        h = self.ln_f(h)
        h_cls = h[:, 0, :]  # [B,C]
        return self.mu(h_cls), self.logvar(h_cls)


class Conductor(nn.Module):
    """
    Deterministic GRU that maps z_p -> z_k sequence of length bars_per_sample.
    """

    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.h0_proj = nn.Linear(cfg.z_dim, cfg.cond_layers * cfg.cond_hidden)
        self.bar_emb = nn.Embedding(cfg.bars_per_sample, cfg.cond_hidden)
        self.gru = nn.GRU(
            input_size=cfg.cond_hidden,
            hidden_size=cfg.cond_hidden,
            num_layers=cfg.cond_layers,
            batch_first=True,
        )

    def forward(self, z_p: torch.Tensor) -> torch.Tensor:
        # z_p: [B,128]
        B = z_p.size(0)
        h0 = self.h0_proj(z_p).view(self.cfg.cond_layers, B, self.cfg.cond_hidden).contiguous()

        bar_ids = torch.arange(self.cfg.bars_per_sample, device=z_p.device)
        x = self.bar_emb(bar_ids)[None, :, :].expand(B, -1, -1)  # [B,bars,384]

        z_seq, _hn = self.gru(x, h0)  # [B,bars,384]
        return z_seq


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embed: int, n_head: int):
        super().__init__()
        assert n_embed % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embed // n_head
        self.qkv = nn.Linear(n_embed, 3 * n_embed, bias=True)
        self.proj = nn.Linear(n_embed, n_embed, bias=True)
        self.dropout = 0.0

    def set_dropout(self, p: float) -> None:
        self.dropout = float(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class FiLMBlock(nn.Module):
    def __init__(self, n_embed: int, n_head: int, d_ff: int, *, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttention(n_embed=n_embed, n_head=n_head)
        self.attn.set_dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.film = nn.Linear(n_embed, 2 * n_embed, bias=True)
        self.ln2 = nn.LayerNorm(n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(n_embed, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, n_embed),
        )
        self.mlp_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        gamma, beta = self.film(cond_tok).chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))
        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x


class FiLMDecoder(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dec_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.dec_n_embed)

        self.blocks = nn.ModuleList(
            [
                FiLMBlock(cfg.dec_n_embed, cfg.dec_n_head, cfg.dec_d_ff, dropout=float(cfg.dec_dropout))
                for _ in range(cfg.dec_n_layers)
            ]
        )
        self.ln_f = nn.LayerNorm(cfg.dec_n_embed)
        self.head = nn.Linear(cfg.dec_n_embed, cfg.vocab_size, bias=True)

    def forward(self, X: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Decoder expects block_size={self.cfg.block_size}, got T={T}")

        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x, cond_tok)
        x = self.ln_f(x)
        return self.head(x)  # [B,T,vocab]


class MusicVAE(nn.Module):
    """
    Conditional VAE: Encoder -> z_p -> Conductor -> z_k; Decoder conditioned via FiLM on C_k=[z_k;A_k].
    """

    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.attr = AttributeEmbedder(cfg)  # [B,8,128]
        self.encoder = BiTransformerEncoder(cfg)
        self.conductor = Conductor(cfg)  # [B,8,384]
        self.decoder = FiLMDecoder(cfg)

        # Project A_k to 128 (already) then concat with z_k(384) -> 512 matches dec_n_embed
        if cfg.cond_hidden + (cfg.attribute_embed_dim * cfg.num_attributes) != cfg.dec_n_embed:
            raise ValueError(
                f"Expected cond_hidden(={cfg.cond_hidden}) + attr_dim(={cfg.attribute_embed_dim*cfg.num_attributes}) "
                f"== dec_n_embed(={cfg.dec_n_embed})"
            )

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    @staticmethod
    def kl_divergence_free_bits(
        mu: torch.Tensor,
        logvar: torch.Tensor,
        *,
        free_bits_lambda_bits: float = 1.0,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Free-bits (hinge) KL to fight posterior collapse (spec).

        Returns:
          - kl_loss: scalar (mean over batch) in nats
          - kl_per_dim_mean: [z_dim] mean KL per dimension in nats
          - active_units: scalar count of dims where KL_dim_mean > lambda_nats
        """
        # KL per dim per sample (nats): [B, z_dim]
        kl_per_dim = -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp())

        # Mean over batch, per dim: [z_dim]
        kl_per_dim_mean = kl_per_dim.mean(dim=0)

        # Spec lambda is in bits; convert to nats.
        lambda_nats = float(free_bits_lambda_bits) * math.log(2.0)

        # Apply free-bits hinge *after* averaging across batch per dimension.
        # This is typically the more stable variant (vs. hinging per-sample).
        kl_per_dim_hinged_mean = torch.clamp(kl_per_dim_mean, min=lambda_nats)  # [z_dim]
        kl_loss = kl_per_dim_hinged_mean.sum()

        active_units = (kl_per_dim_mean > lambda_nats).sum()
        return kl_loss, kl_per_dim_mean, active_units

    def forward(
        self,
        X: torch.Tensor,
        bar_indices: torch.Tensor,
        attributes: torch.Tensor,
        *,
        targets: Optional[torch.Tensor] = None,
        pad_id: Optional[int] = None,
        beta: float = 1.0,
    ) -> dict:
        mu, logvar = self.encoder(X)
        z_p = self.reparameterize(mu, logvar)
        z_k = self.conductor(z_p)  # [B,8,384]

        A_k = self.attr(attributes)  # [B,8,128]
        C_k = torch.cat([z_k, A_k], dim=-1)  # [B,8,512]
        cond_tok = broadcast_by_bar_indices(C_k, bar_indices)  # [B,1024,512]

        logits = self.decoder(X, cond_tok)

        out = {"logits": logits, "mu": mu, "logvar": logvar, "z_p": z_p, "z_k": z_k}

        loss_kl, kl_per_dim_mean, active_units = self.kl_divergence_free_bits(
            mu, logvar, free_bits_lambda_bits=1.0
        )
        out["loss_kl"] = loss_kl
        out["kl_per_dim_mean"] = kl_per_dim_mean
        out["active_units"] = active_units

        if targets is not None:
            if pad_id is None:
                raise ValueError("pad_id is required when computing reconstruction loss.")
            loss_recon = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=int(pad_id),
                reduction="mean",
            )
            out["loss_recon"] = loss_recon
            out["loss_total"] = loss_recon + float(beta) * loss_kl

        return out
'''

DATA_CODE = r'''
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import torch
from torch.utils.data import Dataset


PathLike = str | os.PathLike[str]


@dataclass(frozen=True)
class MemmapSplitMeta:
    split: str
    num_examples: int
    tokens_per_example: int
    bars_per_sample: int
    num_attributes: int
    pad_id: int
    bar_id: int


class MemmapMusicDataset(Dataset):
    def __init__(self, split_dir: PathLike):
        self.split_dir = Path(split_dir)
        payload = json.loads((self.split_dir / 'meta.json').read_text(encoding='utf-8'))

        self.meta = MemmapSplitMeta(
            split=str(payload.get('split', self.split_dir.name)),
            num_examples=int(payload['num_examples']),
            tokens_per_example=int(payload.get('tokens_per_example', payload.get('block_size', 1024))),
            bars_per_sample=int(payload.get('bars_per_sample', 8)),
            num_attributes=4,
            pad_id=int(payload['token_ids']['pad_id']),
            bar_id=int(payload['token_ids']['bar_id']),
        )

        self.x_path = self.split_dir / 'X.u16.memmap'
        self.bi_path = self.split_dir / 'bar_indices.u8.memmap'
        self.a_path = self.split_dir / 'attributes.u8.memmap'

        self._pid: Optional[int] = None
        self._x = None
        self._bi = None
        self._a = None

    def __len__(self) -> int:
        return self.meta.num_examples

    def _ensure_open(self) -> None:
        pid = os.getpid()
        if self._pid == pid and self._x is not None:
            return

        n = self.meta.num_examples
        t = self.meta.tokens_per_example
        b = self.meta.bars_per_sample
        a = self.meta.num_attributes

        self._x = np.memmap(self.x_path, mode='r', dtype=np.uint16, shape=(n, t))
        self._bi = np.memmap(self.bi_path, mode='r', dtype=np.uint8, shape=(n, t))
        self._a = np.memmap(self.a_path, mode='r', dtype=np.uint8, shape=(n, b, a))
        self._pid = pid

    def __getitem__(self, idx: int):
        self._ensure_open()
        return self._x[idx], self._bi[idx], self._a[idx]


def make_collate_fn(*, pad_id: int) -> Callable:
    def collate(batch):
        xs, bis, attrs = zip(*batch)
        x_np = np.stack(xs, axis=0)
        bi_np = np.stack(bis, axis=0)
        a_np = np.stack(attrs, axis=0)

        X = torch.from_numpy(x_np.astype(np.int64, copy=False))
        bar_indices = torch.from_numpy(bi_np.astype(np.int64, copy=False))
        attributes = torch.from_numpy(a_np.astype(np.int64, copy=False))

        Y = torch.empty_like(X)
        Y[:, :-1] = X[:, 1:]
        Y[:, -1] = int(pad_id)

        return {'X': X, 'Y': Y, 'bar_indices': bar_indices, 'attributes': attributes}

    return collate
'''

MODULE_CODE = VAE_CODE + "\n\n" + DATA_CODE

MODULE_PATH.write_text(MODULE_CODE, encoding='utf-8')
print('Wrote module:', MODULE_PATH)

# Import from the written file
import importlib
import musicgen_kaggle_vae_min as mg
print('Imported self-contained module:', mg.__file__)


In [ ]:
# -------------------------
# Locate dataset splits
# -------------------------

def require_dir(p: Path) -> Path:
    if not p.exists() or not p.is_dir():
        raise FileNotFoundError(f"missing directory: {p}")
    return p

def require_file(p: Path) -> Path:
    if not p.exists() or not p.is_file():
        raise FileNotFoundError(f"missing file: {p}")
    return p

BASE = require_dir(Path('/kaggle/input/datasets') / KAGGLE_USER)

TRAIN_ROOT = require_dir(BASE / TRAIN_DATASET)
VAL_ROOT = require_dir(BASE / VAL_DATASET)

TRAIN_DIR = require_dir(TRAIN_ROOT / SPLIT_ROOT_REL)
VAL_DIR = require_dir(VAL_ROOT / SPLIT_ROOT_REL)

for d in [TRAIN_DIR, VAL_DIR]:
    require_file(d / 'meta.json')
    require_file(d / 'X.u16.memmap')
    require_file(d / 'bar_indices.u8.memmap')
    require_file(d / 'attributes.u8.memmap')

print('TRAIN_DIR:', TRAIN_DIR)
print('VAL_DIR:', VAL_DIR)


In [ ]:
# -------------------------
# DataLoader (best-effort de-correlation)
# -------------------------

from torch.utils.data import DataLoader, RandomSampler

train_ds = mg.MemmapMusicDataset(TRAIN_DIR)
val_ds = mg.MemmapMusicDataset(VAL_DIR)

print('train meta:', train_ds.meta)
print('val meta:', val_ds.meta)

pad_id = int(train_ds.meta.pad_id)

# Collate function derives Y=shift(X)+pad and casts to torch.long
collate_fn = mg.make_collate_fn(pad_id=pad_id)

if USE_REPLACEMENT_SAMPLER:
    sampler = RandomSampler(train_ds, replacement=True, num_samples=BATCH_SIZE * MAX_STEPS * GRAD_ACCUM)
    shuffle = False
else:
    sampler = None
    shuffle = True

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    shuffle=shuffle,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS if NUM_WORKERS > 0 else False,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    drop_last=True,
    collate_fn=collate_fn,
)

val_dl = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
    collate_fn=collate_fn,
)

# Quick batch sanity
b = next(iter(train_dl))
print({k: tuple(v.shape) for k, v in b.items()})


In [ ]:
# -------------------------
# Model + optimizer + cosine schedule (10K steps)
# -------------------------

vocab_size = 195

cfg = mg.VAEConfig(
    vocab_size=vocab_size,
    block_size=int(train_ds.meta.tokens_per_example),
    bars_per_sample=int(train_ds.meta.bars_per_sample),
)
model = mg.MusicVAE(cfg).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print('params:', f'{n_params/1e6:.2f}M')

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Cosine annealing over exactly MAX_STEPS optimizer updates
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_STEPS, eta_min=ETA_MIN)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [ ]:
# -------------------------
# Checkpointing (save/resume)
# -------------------------

def save_ckpt(step: int):
    payload = {
        'step': step,
        'model': model.state_dict(),
        'opt': opt.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict() if USE_AMP else None,
        'cfg': cfg.__dict__,
    }
    torch.save(payload, CKPT_PATH)


def try_resume(path: Path | None) -> int:
    if path is None:
        return 0
    path = Path(path)
    if not path.exists():
        print('No checkpoint found at', path, '(starting fresh)')
        return 0

    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model'])
    opt.load_state_dict(ckpt['opt'])
    if 'scheduler' in ckpt and ckpt['scheduler'] is not None:
        scheduler.load_state_dict(ckpt['scheduler'])
    if USE_AMP and ckpt.get('scaler') is not None:
        scaler.load_state_dict(ckpt['scaler'])

    start_step = int(ckpt.get('step', 0))
    print('Resumed from step', start_step)
    return start_step

start_step = try_resume(RESUME_PATH)


In [ ]:
# -------------------------
# Train / Eval loops (ELBO + cyclical β + collapse diagnostics)
# -------------------------

import time
import math
from tqdm.auto import tqdm


def beta_cyclical(step: int) -> float:
    # Spec: cycle_len=10K, ramp 0->0.1 over first 5K, hold 0.1 for next 5K.
    t = int(step) % int(BETA_CYCLE_LEN)
    if t < int(BETA_RAMP_LEN):
        return float(BETA_MAX) * (t / max(int(BETA_RAMP_LEN), 1))
    return float(BETA_MAX)


@torch.no_grad()
def eval_val(max_batches: int = EVAL_BATCHES) -> dict:
    model.eval()
    acc = {
        'loss_total': 0.0,
        'loss_recon': 0.0,
        'loss_kl': 0.0,
        'kl_bits_dim_avg': 0.0,
        'active_units': 0.0,
        'n': 0,
    }

    for i, batch in enumerate(val_dl):
        if i >= max_batches:
            break
        X = batch['X'].to(DEVICE, non_blocking=True)
        Y = batch['Y'].to(DEVICE, non_blocking=True)
        bar_indices = batch['bar_indices'].to(DEVICE, non_blocking=True)
        attributes = batch['attributes'].to(DEVICE, non_blocking=True)

        beta = beta_cyclical(start_step + 1)  # any fixed beta for val (use current cycle position)
        out = model(X, bar_indices, attributes, targets=Y, pad_id=pad_id, beta=beta)

        kl_bits_dim_avg = float((out['kl_per_dim_mean'] / math.log(2.0)).mean().item())

        acc['loss_total'] += float(out['loss_total'].item())
        acc['loss_recon'] += float(out['loss_recon'].item())
        acc['loss_kl'] += float(out['loss_kl'].item())
        acc['kl_bits_dim_avg'] += kl_bits_dim_avg
        acc['active_units'] += float(out['active_units'].item())
        acc['n'] += 1

    n = max(int(acc['n']), 1)
    for k in ['loss_total', 'loss_recon', 'loss_kl', 'kl_bits_dim_avg', 'active_units']:
        acc[k] /= n
    model.train()
    return acc


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h}:{m:02d}:{s:02d}"


def train():
    model.train()

    it = iter(train_dl)

    ema_total = None
    ema_recon = None
    ema_kl = None

    # Track recon improvement during beta≈0 portion of the cycle
    recon_beta0_start = None
    recon_beta0_ema = None
    prev_beta_is_zero = False

    tokens_per_update = BATCH_SIZE * int(train_ds.meta.tokens_per_example) * GRAD_ACCUM

    last_t = time.perf_counter()
    last_step = start_step

    pbar = tqdm(range(start_step + 1, MAX_STEPS + 1), total=MAX_STEPS - start_step, desc='Training')
    for step in pbar:
        opt.zero_grad(set_to_none=True)

        ctx = (
            torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP)
            if DEVICE == 'cuda'
            else torch.autocast('cpu', enabled=False)
        )

        beta = beta_cyclical(step)
        beta_is_zero = beta < 1e-12

        loss_total_sum = 0.0
        loss_recon_sum = 0.0
        loss_kl_sum = 0.0
        kl_bits_dim_avg_sum = 0.0
        active_units_sum = 0.0

        for micro in range(GRAD_ACCUM):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(train_dl)
                batch = next(it)

            X = batch['X'].to(DEVICE, non_blocking=True)
            Y = batch['Y'].to(DEVICE, non_blocking=True)
            bar_indices = batch['bar_indices'].to(DEVICE, non_blocking=True)
            attributes = batch['attributes'].to(DEVICE, non_blocking=True)

            with ctx:
                out = model(X, bar_indices, attributes, targets=Y, pad_id=pad_id, beta=beta)
                loss = out['loss_total'] / GRAD_ACCUM

            scaler.scale(loss).backward()

            loss_total_sum += float(out['loss_total'].item())
            loss_recon_sum += float(out['loss_recon'].item())
            loss_kl_sum += float(out['loss_kl'].item())
            kl_bits_dim_avg_sum += float((out['kl_per_dim_mean'] / math.log(2.0)).mean().item())
            active_units_sum += float(out['active_units'].item())

        scaler.step(opt)
        scaler.update()
        scheduler.step()

        loss_total = loss_total_sum / GRAD_ACCUM
        loss_recon = loss_recon_sum / GRAD_ACCUM
        loss_kl = loss_kl_sum / GRAD_ACCUM
        kl_bits_dim_avg = kl_bits_dim_avg_sum / GRAD_ACCUM
        active_units = active_units_sum / GRAD_ACCUM

        ema_total = loss_total if ema_total is None else (0.95 * ema_total + 0.05 * loss_total)
        ema_recon = loss_recon if ema_recon is None else (0.95 * ema_recon + 0.05 * loss_recon)
        ema_kl = loss_kl if ema_kl is None else (0.95 * ema_kl + 0.05 * loss_kl)

        # Recon delta during beta=0 segment
        if beta_is_zero and not prev_beta_is_zero:
            recon_beta0_start = loss_recon
            recon_beta0_ema = loss_recon
        if beta_is_zero:
            recon_beta0_ema = loss_recon if recon_beta0_ema is None else (0.95 * recon_beta0_ema + 0.05 * loss_recon)
        prev_beta_is_zero = beta_is_zero

        recon_beta0_delta = None
        if beta_is_zero and recon_beta0_start is not None:
            recon_beta0_delta = float(recon_beta0_start - loss_recon)

        lr_now = float(opt.param_groups[0]['lr'])

        if step == start_step + 1 or step % LOG_EVERY == 0:
            now = time.perf_counter()
            dt = now - last_t
            steps_done = step - last_step
            it_s = steps_done / max(dt, 1e-9)
            tok_s = (steps_done * tokens_per_update) / max(dt, 1e-9)
            eta_s = (MAX_STEPS - step) / max(it_s, 1e-9)

            msg = (
                f"step {step}/{MAX_STEPS} | total {loss_total:.4f} (ema {ema_total:.4f})"
                f" | recon {loss_recon:.4f} (ema {ema_recon:.4f})"
                f" | kl {loss_kl:.4f} (ema {ema_kl:.4f})"
                f" | beta {beta:.4f}"
                f" | kl_bits/dim {kl_bits_dim_avg:.3f}"
                f" | active {active_units:.1f}"
                f" | lr {lr_now:.2e}"
                f" | tok/s {tok_s/1e6:.2f}M | eta {fmt_hms(eta_s)}"
            )
            if beta_is_zero and recon_beta0_delta is not None:
                msg += f" | recon_delta(beta0) {recon_beta0_delta:.4f}"
            pbar.set_description(msg)
            tqdm.write(msg)

            last_t = now
            last_step = step

        if step % EVAL_EVERY == 0:
            v = eval_val()
            tqdm.write(
                f"eval @ step {step:5d} | total {v['loss_total']:.4f} | recon {v['loss_recon']:.4f} | kl {v['loss_kl']:.4f}"
                f" | kl_bits/dim {v['kl_bits_dim_avg']:.3f} | active {v['active_units']:.1f}"
            )

        if step % SAVE_EVERY == 0:
            save_ckpt(step)

    save_ckpt(MAX_STEPS)
    print('Training complete. Final checkpoint:', CKPT_PATH)


train()
